## Lab 04

## Zadanie 1 (Igor Sieradzki)

Funkcja <t>flow_rate</i> do modyfikacji

In [4]:
def flow_rate(weight, time, *, period=1, units_per_kg=1):
    """Funkcja wylicza przepływ masy w zadanej jednostce czasu."""
    return weight / time * period * units_per_kg

weight = 0.5
time = 3

In [5]:
flow = flow_rate(weight, time)
print("{0:.3} kg per second".format(flow))

0.167 kg per second


In [17]:
Zmodyfikuj funckje <i>flow_rate</i> tak, aby poniższe wywołania działały poprawnie.

SyntaxError: invalid syntax (3635264092.py, line 1)

In [6]:
flow = flow_rate(weight, time, period=60, units_per_kg=1000)
print("{} grams per minute".format(flow))

10000.0 grams per minute


In [7]:
flow = flow_rate(weight, time, period=1, units_per_kg=1)
print("{0:.3} kg per second".format(flow))

0.167 kg per second


In [8]:
flow = flow_rate(weight, time, period=60, units_per_kg=1000)
print("{0:.3} grams per minute".format(flow))

1e+04 grams per minute


Wyjaśnić czemu poniższe rzucanie błędu jest porządane

ponizsze rzucanie bledu jest porzadane poniewaz zapobiega przekazaniu argumentow w niewlasciwej kolejnosci oraz bledom zwiazanym z uzywaniem nieodpowiednich jednostek. przez dodanie * w sygnaturze funkcji wymuszamy aby wszystkie argumenty wystepujace po niej byly przekazywane jako argumenty nazwane. 

In [9]:
try:
    flow = flow_rate(weight, time, 3600, 2.2)
except TypeError:
    print(True)

True


## Zadanie 2 (Igor Sieradzki)

Dopisz definicje dekoratora _timeit_, który wypisze na wyjscie standardowe czas wywołania udekorowanej funckji np.

```
@timeit
def foo(x):
    return x**2
    
r = foo(2)
```
wypize: <br>
    `Function foo took: 0.00001 seconds` <br>
oraz zwróci 4

In [10]:
from time import time
from functools import wraps

def timeit(func):
    """Wypisuje czas wywołania udekorowanej funkcji."""
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time()
        result = func(*args, **kwargs)
        elapsed = time() - start
        print("Function {} took: {:.5f} seconds".format(func.__name__, elapsed))
        return result
    return wrapper

@timeit
def squares_list(n):
    squares = []
    for i in range(n):
        squares.append(i ** 2)
    return squares

@timeit
def squares_comprehension(n):
    return [i ** 2 for i in range(n)]

@timeit
def squares_map(n):
    return map(lambda x: x**2, range(n))

n = 1000000
l = squares_list(n)
c = squares_comprehension(n)
m = squares_map(n)

Function squares_list took: 0.06574 seconds
Function squares_comprehension took: 0.05439 seconds
Function squares_map took: 0.00000 seconds


## Zadanie 3 (Igor Sieradzki)

Dopisz definicje dekoratora _derivate_ wg. instrukcji w _docstringu_

In [11]:
import sys
sys.float_info.epsilon # epsilon maszynowy


def derivate(epsilon=None):
    """
    Zwraca pochodną funkcji w punkcie, wg. wzoru f'(x) = [f(x+h) - f(x)]/h, 
    gdzie h jest parametrem dekoratora, jeśli nie zostanie podany, należy przyjąć 1000 * epsilon maszynowy
    """
    h = 1000 * sys.float_info.epsilon if epsilon is None else epsilon
    
    def decorator(func):
        from functools import wraps
        @wraps(func)
        def derivative(x):
            return (func(x + h) - func(x)) / h
        return derivative
    return decorator


@derivate(0.01)
def f(x):
    return x*x

@derivate(0.00001)
def g(x):
    return x*x*x+3

def test(a, b, eps=1):
    return abs(round(a)-round(b)) < eps

print(test(f(100), 200.0))
print(round(f(0)) == 0.0)

print(test(g(100), 30000.0))
print(round(g(0)) == 0.0)

from random import random
for x in [random()*1000. for _ in range(20)]:
    print(f(x), 2*x, '\t', test(f(x), 2*x))
    print(g(x), 3*x**2, '\t', test(g(x), 3*x**2))

True
True
True
True
896.8251366051845 896.815136605948 	 True
603208.05311203 603208.0419341589 	 True
668.1713002792094 668.1613002800976 	 True
334829.65156435966 334829.6423939931 	 True
1784.166654094588 1784.1566540992478 	 True
2387411.2486839294 2387411.2247749674 	 True
93.70287982901573 93.69287982902242 	 True
6583.768206473905 6583.766797991727 	 True
1122.0168179657776 1122.0068179651712 	 True
944174.489378929 944174.4746702465 	 True
1834.851616807282 1834.8416168148767 	 True
2524982.8338623047 2524982.819096873 	 True
1213.7989059730899 1213.7889059794022 	 True
1104962.6469612122 1104962.6312090056 	 True
1263.602714391891 1263.5927143954336 	 True
1197499.9248981476 1197499.910904915 	 True
1234.0441938955337 1234.0341938943043 	 True
1142130.3063631058 1142130.293775274 	 True
220.90407383275306 220.8940738325642 	 True
36595.64722329378 36595.64389075975 	 True
338.9570294300938 338.9470294304604 	 True
86163.82163017988 86163.81656980005 	 True
1828.3364154398441 1

## Zadanie 4 (Igor Sieradzki)

Dopisz definicje dekoratora _accepts_ wg. instrukcji w _docstringu_

In [12]:
import inspect
from functools import wraps

def accepts(*types):
    """Sprawdza typy argumentów przekazanych udekorowanej funkcji."""
    def decorator(func):
        signature = inspect.signature(func)
        @wraps(func)
        def wrapper(*args, **kwargs):
            bound = signature.bind(*args, **kwargs)
            values = list(bound.arguments.values())
            if len(values) < len(types):
                raise TypeError("Niepoprawna liczba argumentów")
            for value, expected in zip(values, types):
                if not isinstance(value, expected):
                    raise TypeError("Niepoprawny typ argumentu")
            return func(*args, **kwargs)
        return wrapper
    return decorator


@accepts(str)
def capitalize(word):
    return word[0].upper() + word[1:]

print(capitalize('ola') == 'Ola')

try:
    capitalize(2)
except TypeError:
    print(True)

@accepts(float, int)
def static_pow(base, exp):
    return base ** exp 

print(static_pow(2., 2) == 4.)
print(static_pow(2., exp=2) == 4.)
print(static_pow(base=2., exp=2) == 4.)

try:
    static_pow('x', 10)
except TypeError:
    print(True)
    
try:
    static_pow(2, 2.2)
except TypeError:
    print(True)

True
True
True
True
True
True
True


## Zadanie 5 (Igor Sieradzki)

Dopisz definicje dekoratora _returns_ wg. instrukcji w _docstringu_

In [13]:
def returns(*types):
    """Sprawdza czy udekorowana funkcja zwraca poprawne argumenty, zdefiniowane w parametrach dekoratora"""
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            result = func(*args, **kwargs)
            values = result if isinstance(result, tuple) else (result,)
            if len(values) != len(types) or any(
                not isinstance(value, expected)
                for value, expected in zip(values, types)
            ):
                raise TypeError("Niepoprawny typ wartości zwracanej")
            return result
        return wrapper
    return decorator


@returns(str)
def str_only_identity(word):
    return word

print(str_only_identity('hello') == 'hello')

try:
    str_only_identity(10)
except TypeError:
    print(True)
    
@returns(int, int)
def split_indices(x):
    return x[0], x[1]

print(split_indices(x=[6,9]) == (6,9))

try:
    split_indices('AB')
except TypeError:
    print(True)

True
True
True
True


## Zadanie 6 (Igor Sieradzki)
Stwórz dekorator cached służący do cachowania wywołań dowolnej funkcji, tzn. chcemy by:
* wywołanie funkcji z określonymi argumentami miało miejsce tylko raz
* funkcja mogła przyjmować dowolną liczbę nazwanych i nienazwanych argumentów
* nie musi reagować poprawnie na domyślne argumenty, tzn. wywołanie funkcji z domyślnymi argumentami a podanie dokładnie takich samych może być traktowane jako dwa różne wywołania
* na opakowanej funkcji można wywołać `.cache_reset()`, który usunie cache z pamięci
* wywołanie `.cache_status()` zwraca string z opisem w postaci: <br>
    `Function FUNCTION_NAME called X times, evaluated Y times`

In [14]:
from random import random
from functools import wraps

def cached(func):
    cache = {}
    calls = 0
    evaluations = 0
    @wraps(func)
    def wrapper(*args, **kwargs):
        nonlocal calls, evaluations
        calls += 1
        key = (args, tuple(sorted(kwargs.items())))
        if key not in cache:
            cache[key] = func(*args, **kwargs)
            evaluations += 1
        return cache[key]
    def cache_reset():
        nonlocal cache
        cache = {}
    def cache_status():
        return "Function {} called {} times, evaluated {} times".format(
            func.__name__, calls, evaluations
        )
    wrapper.cache_reset = cache_reset
    wrapper.cache_status = cache_status
    return wrapper

@cached
def foo(x, y=1, z=4):
    return random()
        
print(foo(3) == foo(3))
print(foo(4) == foo(4))
print(foo(3, z=-1, y=3) == foo(3, y=3, z=-1))
print(foo(3) != foo(x=3))
a = foo(3)
foo.cache_reset()
print(a != foo(3))
print(foo.cache_status() == 'Function foo called 10 times, evaluated 5 times')

True
True
True
True
True
True


## Zadanie 7 (Krzysztof Hajto)

Napisz dekorator który będzie robić n-krotne złożenie funkcji, gdzie n jest parametrem dekoratora

In [15]:
def zlozenie(n):
    def decorator(func):
        from functools import wraps
        @wraps(func)
        def wrapper(*args, **kwargs):
            result = func(*args, **kwargs)
            for _ in range(n - 1):
                result = func(result)
            return result
        return wrapper
    return decorator

@zlozenie(3)
def f1(x):
    return x+1

@zlozenie(2)
def f2(x):
    return x*x

@zlozenie(5)
def f3(word):
    return "".join(chr(ord(l)+1) for l in word)

print(f1(2)==5)
print(f2(3)==81)
print(f3("alamakota")=="fqfrfptyf")

True
True
True


## Zadanie 8 (Krzysztof Hajto)

Python nie ma wbudowanej instrukcji switch. Ale posiada anonimowe funkcje oraz słowniki. Zaimplementuj poniższy switch w postaci słownika funkcji.
`
int my_function(x, y) {


    switch(x) {
        case 1: return y*y;
        case 2: return x+y;
        case 3: return x*y;
        case 4: return 0;
    }
}
`

PS. Nigdy nie róbcie tego w faktycznym kodzie :)

In [16]:
def my_function(x, y):
    functions = {
        1: lambda: y * y,
        2: lambda: x + y,
        3: lambda: x * y,
        4: lambda: 0,
    }
    return functions[x]()

print(my_function(1,3)==9)
print(my_function(2,4)==6)
print(my_function(3,1)==3)
print(my_function(4,9)==0)

True
True
True
True
